In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import numpy as np
import pandas as pd
import os

In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

print(f"Train: {train.shape}, Test: {test.shape}")
train.head()

Train: (2000, 8), Test: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
most_common = train['answer'].value_counts().idxmax()
print(f"Most common answer: {most_common}")

sub['answer'] = most_common
print(sub.head())

Most common answer: B
   ID Prediction answer
0   1      A B C      B
1   2      A B C      B
2   3      A B C      B
3   4      A B C      B
4   5      A B C      B


In [8]:
# Load fresh sample submission
sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

print("Sample submission columns:", sub.columns.tolist())
print(sub.head())

Sample submission columns: ['ID', 'Prediction']
   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C


In [9]:
# Use the correct column name from sample_submission
sub['Prediction'] = most_common   # replace 'Prediction' with whatever column name printed above

sub.to_csv('submission.csv', index=False)
print("Done!")
print(sub.head())

Done!
   ID Prediction
0   1          B
1   2          B
2   3          B
3   4          B
4   5          B


# Milestone 1:-

In [1]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

### Question 1: Answer Distribution
Calculate the frequency distribution of the correct answer (A, B, C, D, E) in `train.csv`. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [2]:
counts = df['answer'].value_counts()
sum_most_least = counts.max() + counts.min()
sum_most_least

814

### Question 2: Vocabulary Size

In [3]:
def clean_text(text):
    text = text.lower()
    return text.translate(str.maketrans('', '', string.punctuation))

cleaned_prompts = df['prompt'].apply(clean_text)
vocab = set(' '.join(cleaned_prompts).split())
len(vocab)

859

### Question 3: Stop Word Filtering

In [4]:
row1_tokens = clean_text(df.iloc[0]['prompt']).split()
filtered_tokens = [w for w in row1_tokens if w not in ENGLISH_STOP_WORDS]
len(filtered_tokens)

13

### Question 4: TF-IDF Vocabulary

In [6]:
combined_text = df['prompt'] + " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(combined_text)
len(vectorizer.get_feature_names_out())

2762

### Question 5: Cosine Similarity

In [7]:
prompt_vec = vectorizer.transform([df.iloc[0]['prompt']])
option_a_vec = vectorizer.transform([df.iloc[0]['A']])
round(cosine_similarity(prompt_vec, option_a_vec)[0][0], 4)

np.float64(0.272)

### Question 6: Similarity Matching Accuracy

In [8]:
def get_best_option(row):
    options = ['A', 'B', 'C', 'D', 'E']
    prompt_v = vectorizer.transform([row['prompt']])
    scores = {opt: cosine_similarity(prompt_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return max(scores, key=scores.get)

matches = df.apply(lambda row: get_best_option(row) == row['answer'], axis=1)
matches.mean() * 100

np.float64(13.55)

### Question 7 & 8: MAP@3 Scoring Functions

In [9]:
def map_at_3(truth, predictions):
    if truth in predictions[:3]:
        rank = predictions.index(truth) + 1
        return 1 / rank
    return 0

### Question 9: Majority Class Baseline

In [10]:
top_answers = counts.index.tolist()[:3]
scores = df['answer'].apply(lambda truth: map_at_3(truth, top_answers))
scores.mean()

np.float64(0.42125)

### Question 10: TF-IDF Pipeline Evaluation

In [11]:
def get_top3_preds(row):
    options = ['A', 'B', 'C', 'D', 'E']
    p_v = vectorizer.transform([row['prompt']])
    s = {opt: cosine_similarity(p_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return sorted(s, key=s.get, reverse=True)

all_scores = df.apply(lambda row: map_at_3(row['answer'], get_top3_preds(row)), axis=1)
all_scores.mean()

np.float64(0.2961666666666667)